## 1) Get Imports and Load Data

In [8]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

import joblib # For saving the model
import os

print("✅ Pipeline imports ready")

✅ Pipeline imports ready


## 2) Load Merged Data & Encode Ethnicity

In [3]:
df = pd.read_csv("../data/processed/cleaned_merged.csv")

print(f"📊 Loaded {len(df)} rows")
print(f"   Columns: {list(df.columns)}")

# Create ethnicity dummies
df = pd.get_dummies(df, columns=["RIDRETH1"], prefix="eth", drop_first=True)

# Convert boolean dummies to int (0/1)
eth_columns = [col for col in df.columns if col.startswith("eth_")]
for col in eth_columns:
    df[col] = df[col].astype(int)

print(f"✅ Ethnicity dummies created: {eth_columns}")
print(f"   Total columns: {list(df.columns)}")

📊 Loaded 7452 rows
   Columns: ['SEQN', 'BPXOSY1', 'BPXODI1', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'BMXBMI']
✅ Ethnicity dummies created: ['eth_2.0', 'eth_3.0', 'eth_4.0', 'eth_5.0']
   Total columns: ['SEQN', 'BPXOSY1', 'BPXODI1', 'RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'eth_2.0', 'eth_3.0', 'eth_4.0', 'eth_5.0']


## 3) Define Features and Create Pipeline

In [4]:
# Features that need scaling (large numerical values)
numeric_features = ["RIDAGEYR", "BMXBMI"]

# Features that are already 0/1 (no scaling needed)
categorical_features = ["RIAGENDR", "eth_2.0", "eth_3.0", "eth_4.0", "eth_5.0"]

# All feature columns (in the order the model expects)
feature_columns = numeric_features + categorical_features

print(f"Numeric (will be scaled): {numeric_features}")
print(f"Categorical (passthrough): {categorical_features}")
print(f"All features: {feature_columns}")

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", "passthrough", categorical_features)
    ]
)

print("✅ Preprocessor created")

Numeric (will be scaled): ['RIDAGEYR', 'BMXBMI']
Categorical (passthrough): ['RIAGENDR', 'eth_2.0', 'eth_3.0', 'eth_4.0', 'eth_5.0']
All features: ['RIDAGEYR', 'BMXBMI', 'RIAGENDR', 'eth_2.0', 'eth_3.0', 'eth_4.0', 'eth_5.0']
✅ Preprocessor created


## 4) Prepare X and y

In [5]:
# Features: age, gender, BMI, ethnicity dummies (Features (X) and target (Y))
X = df[feature_columns] # Features (raw values)
y = df["BPXOSY1"] # Target (raw systolic BP)

# Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Data shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train range: {y_train.min():.0f} - {y_train.max():.0f} mmHg")
print(f"   y_test range:  {y_test.min():.0f} - {y_test.max():.0f} mmHg")

📊 Data shapes:
   X_train: (5961, 7)
   X_test:  (1491, 7)
   y_train range: 61 - 225 mmHg
   y_test range:  77 - 210 mmHg


## 5) Train the Pipeline

In [6]:
# Build pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train Model
pipeline.fit(X_train, y_train)
print("✅ Pipeline trained successfully!")

# Show coefficients
print(f"\n📈 Model coefficients:")
for name, coef in zip(feature_columns, pipeline.named_steps['regressor'].coef_):
    print(f"   {name}: {coef:.4f}")
print(f"   Intercept: {pipeline.named_steps['regressor'].intercept_:.4f}")

✅ Pipeline trained successfully!

📈 Model coefficients:
   RIDAGEYR: 9.4557
   BMXBMI: 0.7099
   RIAGENDR: -4.9424
   eth_2.0: -0.2193
   eth_3.0: -0.9593
   eth_4.0: 3.8828
   eth_5.0: 0.6868
   Intercept: 126.7231


## 6) Predict and Evaluate

In [10]:
# Predict
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

# Metrics
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("📊 Model Performance:")
print(f"   Train MSE: {train_mse:.2f}")
print(f"   Test MSE:  {test_mse:.2f}")
print(f"   Train R²:  {train_r2:.4f}")
print(f"   Test R²:   {test_r2:.4f}")
print(f"   Test RMSE: {np.sqrt(test_mse):.2f} mmHg")

📊 Model Performance:
   Train MSE: 243.35
   Test MSE:  239.57
   Train R²:  0.2857
   Test R²:   0.3092
   Test RMSE: 15.48 mmHg


## 7) Save Pipeline (Model)

In [26]:
# Create models directory
os.makedirs("../models", exist_ok=True)

# Save pipeline
pipeline_path = "../models/bp_pipeline.pkl"
joblib.dump(pipeline, pipeline_path)

# Save feature info for reference
feature_info = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "all_features": feature_columns
}
joblib.dump(feature_info, "../models/feature_info.pkl")

print(f"✅ Pipeline saved to: {pipeline_path}")
print(f"✅ Feature info saved")
print(f"   File size: {os.path.getsize(pipeline_path) / 1024:.1f} KB")

✅ Pipeline saved to: ../models/bp_pipeline.pkl
✅ Feature info saved
   File size: 2.8 KB


## 8) Test the Pipeline

In [27]:
# Load pipeline
loaded_pipeline = joblib.load("../models/bp_pipeline.pkl")

# Test sample (RAW values, no manual scaling!)
test_sample = pd.DataFrame([{
    "RIDAGEYR": 50,      # Raw age
    "RIAGENDR": 1,       # Male
    "BMXBMI": 28.0,      # Raw BMI
    "eth_2.0": 0,
    "eth_3.0": 1,        # White
    "eth_4.0": 0,
    "eth_5.0": 0
}])

# Predict
prediction = loaded_pipeline.predict(test_sample)[0]

print(f"🎯 Test prediction: {prediction:.1f} mmHg")
print(f"   Input: Age=50, Male, BMI=28, White")

🎯 Test prediction: 122.9 mmHg
   Input: Age=50, Male, BMI=28, White


---
## 9) Prediction Confidence / Error Rate

### Model Residuals

In [28]:
y_pred_train = pipeline.predict(X_train)
residuals = y_train - y_pred_train
sigma_residuals = np.std(residuals)
print(sigma_residuals)

# Save file as .pkl
joblib.dump(sigma_residuals, "../models/sigma_residuals.pkl")

print("sigma_residuals:", sigma_residuals)  # Should be a single number like 0.85
print("type:", type(sigma_residuals))       # Should be <class 'numpy.float64'>

15.599699043961873
sigma_residuals: 15.599699043961873
type: <class 'numpy.float64'>


### Training Samples (n) and Number of Features (p)
- n: The total number of rows in your training dataset.
- p: The number of input features (columns) your model was trained on.

In [29]:
n_train = X_train.shape[0]
p_features = X_train.shape[1]
print(n_train)
print(n_train)

# Save n and p for degrees of freedom
joblib.dump({"n": n_train, "p": p_features}, "../models/training_info.pkl")

5961
5961


['../models/training_info.pkl']

### t-critical value

In [30]:
import scipy

t_val = scipy.stats.t.ppf(0.975, df)
print(t_val)

[[ 1.95998218  1.97769228  1.98446745 ...         nan         nan
  12.70620474]
 [ 1.95998218  1.97976376  1.98860967 ... 12.70620474         nan
          nan]
 [ 1.95998218  1.98156676  1.99045021 ...         nan         nan
          nan]
 ...
 [ 1.95998065  1.98259726  1.99713791 ...         nan         nan
          nan]
 [ 1.95998065  1.97881953  1.98968632 ...         nan         nan
          nan]
 [ 1.95998065  1.97867085  1.98968632 ... 12.70620474         nan
          nan]]


---
## 10) Out-of-Distribution (OOD) Detection using Mahalanobis Distance

The Mahalanobis distance measures how far a new point is from the center of the training data distribution. The mean vector is that center.

### Mean Vector of Scaled Training Data (μ)

A vector (list of numbers) containing the mean of each feature in the scaled training data. `mean_vector = np.mean(X_train_scaled, axis=0)`

In [31]:
mean_vector = np.mean(X_train.values, axis=0)

print(mean_vector)
print(type(mean_vector))

# Save file as .pkl
joblib.dump(mean_vector, "../models/mean_vector.pkl")

[44.97970139 28.30979701  1.53967455  0.11172622  0.55124979  0.12263043
  0.12799866]
<class 'numpy.ndarray'>


['../models/mean_vector.pkl']

### Covariance Matrix of the Scaled Training Data (Σ)

A matrix that describes the variance of each feature and the covariance between features in the scaled training data. `cov_matrix = np.cov(X_train_scaled, rowvar=False)`

The Mahalanobis distance uses the inverse of this matrix to account for correlations between features. A feature that is highly correlated with another should not contribute as much "new" information.

In [32]:
cov_matrix = np.cov(X_train, rowvar=False)
print(f"cov_matrix shape: {cov_matrix.shape}")

# Save the inverse directly
cov_matrix_inv = np.linalg.inv(cov_matrix)
print(f"cov_matrix_inv shape: {cov_matrix_inv.shape}")

joblib.dump(cov_matrix_inv, "../models/cov_matrix_inv.pkl")


cov_matrix shape: (7, 7)
cov_matrix_inv shape: (7, 7)


['../models/cov_matrix_inv.pkl']

### The OOD Threshold (χ² value)

This is the decision boundary and it is a single number. If the Mahalanobis distance of a new point is greater than this threshold, the point is flagged as "out-of-distribution."

You compute it in app.py using the chi-squared distribution.
`threshold = chi2.ppf(0.95, df=p) (for a 95% confidence level)`